# scoutfield 01 Perception Finetune

Leakage check, then fine-tune EfficientNet-B0 on PlantVillage.

Roadmap item 1. Writes `leakage_report.json`, `best.pt`, `last.pt` and `perception_summary.json` to stage `01_perception` of the artefact store. `last.pt` is pushed every epoch.

If the session dies part-way through training, set `RESUME = True` in the run cell and run again: it continues from the last finished epoch.

> Logic lives in `scoutfield/` and `experiments/`. This notebook orchestrates: install, configure, call, display.


In [ ]:
# ------------------------------------------------------------------ checks
# Run before cloning or installing anything, so a misconfigured notebook stops in
# seconds and spends almost no quota.
import os

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    raise RuntimeError("Attach the HF_TOKEN secret: Add-ons → Secrets") from exc
if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("Attach the HF_TOKEN secret: Add-ons → Secrets")

import torch

if not torch.cuda.is_available() or torch.cuda.get_device_capability() < (7, 0):
    raise RuntimeError("Set the accelerator to GPU T4 x2")

# ---------------------------------------------------------------- bootstrap
# Requires: Internet ON, the HF_TOKEN secret attached, and the accelerator in the
# notebook's settings (GPU T4 x2).
#
# The repo is CLONED, not merely pip-installed. `pyproject.toml` ships the
# `scoutfield*` packages only, so `experiments/` and `configs/` would otherwise be
# missing and every run cell below would fail with "can't open file".
import shutil
import subprocess
import sys

REPO = "https://github.com/OSegun/scoutfield.git"
BRANCH = "main"
SRC = "/kaggle/working/scoutfield"
os.environ["SCOUTFIELD_KAGGLE_KERNEL"] = "odusinaoluwasegun/scoutfield-01-perception-finetune"

# Cloned if absent, reset to the branch tip if present: a clone left over in
# /kaggle/working would otherwise pin the session to an old commit.
if not os.path.isdir(os.path.join(SRC, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, SRC],
                   check=True)
else:
    subprocess.run(["git", "-C", SRC, "fetch", "--depth", "1", "origin", BRANCH],
                   check=True)
    subprocess.run(["git", "-C", SRC, "reset", "--hard", f"origin/{BRANCH}"], check=True)
    subprocess.run(["git", "-C", SRC, "clean", "-fd"], check=True)
os.chdir(SRC)

# Dependencies come from the CLONED file, including huggingface_hub for the
# artefact store. Kaggle already ships torch, torchvision, numpy, pandas,
# scikit-learn and pillow, so they are not reinstalled.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                "requirements-kaggle.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"],
               check=True)

# `utils.paths` writes to /kaggle/working, while the configs name repo-relative
# paths such as results/_done.json. Linking the repo's results/, checkpoints/ and
# figures/ to /kaggle/working makes both name the same directory.
for _d in ("results", "checkpoints", "figures"):
    _target = f"/kaggle/working/{_d}"
    os.makedirs(_target, exist_ok=True)
    if os.path.islink(_d):
        continue
    if os.path.isdir(_d):
        shutil.rmtree(_d)          # a fresh clone puts only .gitkeep in here
    os.symlink(_target, _d)

import scoutfield
from scoutfield.utils.artifacts import load_settings

COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
SETTINGS = load_settings()
print("repo commit:", COMMIT, "| scoutfield", scoutfield.__version__,
      "| pilot pinned at", scoutfield.PILOT_TAG)
print("run_id:", SETTINGS["run_id"])
print("artefact store:", SETTINGS["hub_repo"])
print("HF_TOKEN:", "present" if os.environ.get("HF_TOKEN") else "MISSING")

## Run


In [ ]:
RESUME = False   # True continues this run from 01_perception/last.pt

!python experiments/00_check_leakage.py --config configs/perception.yaml
assert _exit_code == 0, 'leakage check failed; see the output above'

RESUME_FLAG = '--resume' if RESUME else ''
!python experiments/01_finetune_perception.py --config configs/perception.yaml $RESUME_FLAG
assert _exit_code == 0, 'fine-tuning failed; see the output above'